# Comparación de modelos de BCS: características congeladas contra CNN afinada

En esta libreta comparamos cinco enfoques para estimar la condición corporal (BCS). Todos usan los mismos 604 recortes y los mismos folds por animal.

| Experimento | Modelo | Pregunta |
|---|---|---|
| E0 | Predecir siempre el promedio del entrenamiento | Referencia: cuánto aportan los modelos en realidad |
| E1 | ResNet-50 afinada (receta 05 de `Modelo_CNN_Vacuno.ipynb`) | Es el modelo de la iteración anterior |
| E2 | ResNet-50 congelada + Ridge | ¿Hace falta afinar la red o bastan sus características? |
| E3 | DINOv2 (ViT-B y ViT-L) congelado + Ridge | ¿Un extractor autosupervisado más grande da mejores características? |
| E4 | DINOv2-L congelado + cabeza densa (MLP) | ¿Una cabeza con más capacidad mejora a Ridge? |

Usamos características congeladas porque, con solo 80 animales, la CNN afinada memoriza a las vacas (MAE de entrenamiento de 0.16 contra 0.30 en prueba). Si congelamos el extractor y solo ajustamos una regresión Ridge, quedan muy pocos parámetros por aprender, que es lo que conviene con un conjunto tan pequeño.

Para no depender de un solo sorteo, además de los folds de la iteración anterior (semilla 42) repetimos todo con otras 5 semillas de folds.

### Fases
1. Configuración y datos (mismos filtros y folds que `Modelo_CNN_Vacuno.ipynb`).
2. Extracción de características congeladas (una sola vez, en caché).
3. Validación cruzada de E0 a E4 para cada semilla de folds.
4. Métricas, intervalos de confianza y criterios de éxito.

### Referencias
- Oquab et al. (2023), DINOv2: Learning Robust Visual Features without Supervision. https://arxiv.org/abs/2304.07193
- Modelos preentrenados de `timm`: https://huggingface.co/timm

## 1. Configuración

In [ ]:
from pathlib import Path
import numpy as np

# Rutas: carpeta con recortes/recortes.csv (salida de SAM.ipynb) y la tabla de imágenes (libreta 01 v2)
EN_COLAB = False
if EN_COLAB:
    DATOS = Path('/content/datos')
    RUTA_TABLA = Path('/content/drive/MyDrive/lecheros/Dataset/imagenes.csv')
else:
    DATOS = Path.home() / 'bcs_data' / 'nube' / 'datos'
    RUTA_TABLA = Path.home() / 'bcs_data' / 'imagenes.csv'
CACHE = DATOS.parent / 'cache_caracteristicas'    # fuera del repositorio: son derivadas de fotos del CAETEC

# Validación
K_FOLDS = 5
SEMILLAS_FOLDS = [42, 1, 2, 3, 4, 5]               # 42 = folds de Modelo_CNN_Vacuno.ipynb
SEMILLA = 42

# Extractores congelados: (nombre corto, modelo de timm, tamaño de entrada)
EXTRACTORES = {
    'resnet50_224': ('resnet50.a1_in1k', 224),
    'dinov2_b_448': ('vit_base_patch14_dinov2.lvd142m', 448),
    'dinov2_l_224': ('vit_large_patch14_dinov2.lvd142m', 224),
    'dinov2_l_448': ('vit_large_patch14_dinov2.lvd142m', 448),
}
VARIANTE = 'archivo_mascara_clahe'                  # recorte con máscara SAM + CLAHE
ALPHAS_RIDGE = list(np.logspace(-1, 5, 25))
ALPHAS_MLP = [0.01, 0.1, 1.0, 10.0, 100.0]

# E1 (CNN afinada): tarda ~4 min por semilla en una GPU; se puede apagar
CORRER_E1 = True
EPOCAS, LOTE = 15, 32
LR_CAPAS, LR_CABEZA, WEIGHT_DECAY, DROPOUT = 1e-4, 1e-3, 1e-2, 0.2
AUGMENTATION = dict(volteo=0.5, giro_grados=8, recorte_min=0.90, brillo=0.15, contraste=0.15)

# Objetivos del proyecto (Business Understanding)
INTERVALO, OBJ_MAE, OBJ_RMSE, OBJ_TOLERANCIA = (3.25, 3.50), 0.25, 0.35, 80.0
PASO, ESCALA, BOOTSTRAP = 0.25, (1.0, 5.0), 5000

## 2. Librerías y dispositivo

In [ ]:
import math, re, time, warnings
import numpy as np, pandas as pd, timm, torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

DISPOSITIVO = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
CACHE.mkdir(parents=True, exist_ok=True)
print('Dispositivo:', DISPOSITIVO)

## 3. Datos y folds

Aplicamos los mismos filtros que en `Modelo_CNN_Vacuno.ipynb`: recortes `ok` del rancho, sin vistas laterales y con `usar = TRUE` en la tabla. Las columnas `animal` y `ronda` vienen de la tabla. Una sesión es una vaca en una visita (`id_vaca`) y un animal es el arete, así que una vaca que se fotografió en dos visitas tiene dos sesiones del mismo animal.

Los folds se arman por animal: barajamos los animales una vez con la semilla y los repartimos en 5 grupos. Con la semilla 42 salen exactamente los mismos folds que en la iteración anterior.

In [ ]:
def a_bool(v):
    return str(v).strip().lower() in {'true', '1', 'verdadero', 'si', 'sí', 'yes', 'x'}

def a_relativa(r):
    m = re.search(r'(?:^|/)(clean/.+)$', str(r).replace('\\', '/'))
    return m.group(1) if m else None

def limpiar_id(s):
    return s.astype(str).str.strip().str.replace(r'\.0$', '', regex=True)

rec = pd.read_csv(DATOS / 'recortes' / 'recortes.csv', dtype={'id_vaca': str, 'animal': str})
tab = pd.read_csv(RUTA_TABLA)
tab['archivo_limpio'] = tab['archivo_limpio'].map(a_relativa)
tab = tab.dropna(subset=['archivo_limpio']).drop_duplicates('archivo_limpio')
tab['animal'], tab['usar_tabla'] = limpiar_id(tab['animal']), tab['usar'].map(a_bool)
rec = rec.drop(columns=['animal', 'ronda'], errors='ignore').merge(
    tab[['archivo_limpio', 'animal', 'ronda', 'usar_tabla']], on='archivo_limpio', how='left')
rec['id_vaca'] = limpiar_id(rec['id_vaca'])
datos = rec[(rec['estado'] == 'ok') & (rec['fuente'] == 'rancho') & (rec['vista'].fillna('') != 'lateral')
            & (rec['usar_tabla'] == True)].reset_index(drop=True).copy()
datos['bcs'] = datos['bcs'].astype(float)

def folds(semilla):
    animales = sorted(datos['animal'].unique())
    orden = np.random.default_rng(semilla).permutation(animales)
    asignacion = {a: i % K_FOLDS + 1 for i, a in enumerate(orden)}
    f = datos['animal'].map(asignacion).to_numpy()
    assert pd.Series(f).groupby(datos['animal']).nunique().max() == 1, 'un animal quedó en dos folds'
    return f

FOLDS = {s: folds(s) for s in SEMILLAS_FOLDS}
Y = datos['bcs'].to_numpy()
ANIMAL = datos['animal'].to_numpy()
PESO = (1 / datos.groupby('id_vaca')['bcs'].transform('size')).to_numpy()   # cada sesión pesa 1 en total

print(f"{len(datos)} fotos · {datos['animal'].nunique()} animales · {datos['id_vaca'].nunique()} sesiones")
display(datos.assign(fold=FOLDS[42]).groupby('fold').agg(animales=('animal', 'nunique'), sesiones=('id_vaca', 'nunique'),
                                                         fotos=('bcs', 'size')))

## 4. Características congeladas

Cada recorte se redimensiona, se normaliza con las medias de ImageNet y pasa por el extractor sin entrenarlo. Promediamos las características de la foto y de su versión en espejo.

- ResNet-50: promedio global de la última capa (2048 valores).
- DINOv2: token `CLS` concatenado con el promedio de los parches (2 × 768 en ViT-B y 2 × 1024 en ViT-L). DINOv2 trabaja con parches de 14 px, así que a 448 px la cuadrícula es de 32 × 32 y se conserva más detalle de la cadera y la base de la cola.

Las características se guardan en caché (`.npz`) fuera del repositorio. Así la extracción se hace una sola vez y todas las semillas usan las mismas.

In [ ]:
MEDIA = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
DESV = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

@torch.no_grad()
def caracteristicas(modelo, es_vit, x):
    if not es_vit:
        return modelo(x)
    t = modelo.forward_features(x)                      # [lote, 1 + parches, D]
    return torch.cat([t[:, 0], t[:, 1:].mean(1)], 1)    # CLS + promedio de parches

def extraer(corto, lote=16):
    archivo = CACHE / f'{corto}_{VARIANTE}.npz'
    if archivo.exists():
        z = np.load(archivo, allow_pickle=True)
        if (z['archivo'] == datos['archivo_limpio'].to_numpy()).all():
            return z['X']
    nombre, tam = EXTRACTORES[corto]
    es_vit = 'dinov2' in nombre
    modelo = timm.create_model(nombre, pretrained=True, num_classes=0, **({'img_size': tam} if es_vit else {}))
    modelo = modelo.eval().to(DISPOSITIVO)
    salida, inicio = [], time.time()
    for i in range(0, len(datos), lote):
        imgs = [np.asarray(Image.open(DATOS / r).convert('RGB').resize((tam, tam), Image.BICUBIC))
                for r in datos[VARIANTE].iloc[i:i + lote]]
        x = ((torch.from_numpy(np.stack(imgs)).permute(0, 3, 1, 2).float() / 255 - MEDIA) / DESV).to(DISPOSITIVO)
        f = (caracteristicas(modelo, es_vit, x) + caracteristicas(modelo, es_vit, torch.flip(x, dims=[3]))) / 2
        salida.append(f.float().cpu().numpy())
    X = np.concatenate(salida)
    np.savez(archivo, X=X, archivo=datos['archivo_limpio'].to_numpy())
    print(f'{corto}: {X.shape} en {time.time() - inicio:.0f} s')
    del modelo
    return X

CARACT = {corto: extraer(corto) for corto in EXTRACTORES}
display(pd.DataFrame({c: {'dimensiones': X.shape[1]} for c, X in CARACT.items()}).T)

## 5. Modelos sobre características congeladas (E0, E2, E3, E4)

- E0 (constante): predice el promedio de las sesiones de entrenamiento.
- Ridge (E2, E3): estandarizamos las características y ajustamos una regresión lineal con penalización L2. Cada sesión pesa 1 en total, repartido entre sus fotos.
- MLP (E4): PCA a 256 componentes, una capa oculta de 256 neuronas y penalización L2.

La fuerza de la penalización (`alpha`) se elige solo con los datos de entrenamiento de cada vuelta, usando 4 folds internos por animal y el MAE ponderado por sesión. El fold de prueba nunca entra en esa elección.

In [ ]:
def mae_ponderado(y, p, w):
    return np.sum(w * np.abs(y - p)) / np.sum(w)

def ajustar(tipo, X, idx, alpha):
    mu = np.average(Y[idx], weights=PESO[idx])
    if tipo == 'ridge':
        m = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
        m.fit(X[idx], Y[idx] - mu, ridge__sample_weight=PESO[idx])
    else:
        m = make_pipeline(StandardScaler(), PCA(256, random_state=SEMILLA),
                          MLPRegressor(hidden_layer_sizes=(256,), alpha=alpha, max_iter=300, random_state=SEMILLA))
        m.fit(X[idx], Y[idx] - mu, mlpregressor__sample_weight=PESO[idx])
    return lambda Xn: m.predict(Xn) + mu

def elegir_alpha(tipo, X, idx):
    animales = np.unique(ANIMAL[idx])
    interno = {a: i % 4 for i, a in enumerate(np.random.default_rng(SEMILLA).permutation(animales))}
    g = np.array([interno[a] for a in ANIMAL[idx]])
    alphas = ALPHAS_RIDGE if tipo == 'ridge' else ALPHAS_MLP
    errores = [np.mean([mae_ponderado(Y[idx[g == k]], ajustar(tipo, X, idx[g != k], a)(X[idx[g == k]]), PESO[idx[g == k]])
                        for k in range(4)]) for a in alphas]
    return alphas[int(np.argmin(errores))]

def validar(tipo, X, fold):
    pred = np.zeros(len(datos))
    for k in range(1, K_FOLDS + 1):
        tr, te = np.where(fold != k)[0], np.where(fold == k)[0]
        pred[te] = ajustar(tipo, X, tr, elegir_alpha(tipo, X, tr))(X[te])
    return pred

def constante(fold):
    pred = np.zeros(len(datos))
    for k in range(1, K_FOLDS + 1):
        pred[fold == k] = datos[fold != k].groupby('id_vaca')['bcs'].first().mean()
    return pred

MODELOS = {
    'E0 constante': lambda f: constante(f),
    'E2 ResNet-50 congelada + Ridge': lambda f: validar('ridge', CARACT['resnet50_224'], f),
    'E3 DINOv2-B 448 + Ridge': lambda f: validar('ridge', CARACT['dinov2_b_448'], f),
    'E3 DINOv2-L 224 + Ridge': lambda f: validar('ridge', CARACT['dinov2_l_224'], f),
    'E3 DINOv2-L 448 + Ridge': lambda f: validar('ridge', CARACT['dinov2_l_448'], f),
    'E4 DINOv2-L 448 + MLP': lambda f: validar('mlp', CARACT['dinov2_l_448'], f),
}

## 6. E1: CNN afinada (receta 05 de `Modelo_CNN_Vacuno.ipynb`)

Es la misma receta de la iteración anterior: ResNet-50 preentrenada donde solo se entrenan `layer4` y la cabeza, augmentation en la GPU, 15 épocas, pérdida cuadrática ponderada por sesión y la predicción final es el promedio de la foto y su espejo. La volvemos a correr aquí para tener predicciones pareadas con los demás modelos (mismas fotos y mismos folds) y así poder calcular intervalos de confianza de la diferencia.

La precisión mixta solo se usa en CUDA. En MPS o CPU se entrena en fp32, por lo que los números pueden variar en milésimas respecto a los de Colab.

In [ ]:
USAR_AMP = DISPOSITIVO == 'cuda'
ENTRENABLES = ('layer4', 'fc')
if CORRER_E1:
    IMGS = torch.from_numpy(np.stack([np.asarray(Image.open(DATOS / r).convert('RGB').resize((224, 224), Image.BICUBIC))
                                      for r in datos[VARIANTE]])).permute(0, 3, 1, 2).contiguous()
MEDIA_D, DESV_D = MEDIA.to(DISPOSITIVO), DESV.to(DISPOSITIVO)

def aumentar(x, gen):
    B, A = x.shape[0], AUGMENTATION
    u = lambda: torch.rand(B, generator=gen)
    voltear = u() < A['volteo']
    x = torch.where(voltear.to(x.device).view(B, 1, 1, 1), torch.flip(x, dims=[3]), x)
    ang = (u() * 2 - 1) * A['giro_grados'] * math.pi / 180
    esc = A['recorte_min'] + u() * (1 - A['recorte_min'])
    tx, ty = (u() * 2 - 1) * (1 - esc), (u() * 2 - 1) * (1 - esc)
    theta = torch.stack([torch.stack([torch.cos(ang) * esc, -torch.sin(ang) * esc, tx], 1),
                         torch.stack([torch.sin(ang) * esc, torch.cos(ang) * esc, ty], 1)], 1).to(x.device, x.dtype)
    x = F.grid_sample(x, F.affine_grid(theta, x.shape, align_corners=False), mode='bilinear', padding_mode='border', align_corners=False)
    brillo = (1 + (u() * 2 - 1) * A['brillo']).to(x.device).view(B, 1, 1, 1)
    contraste = (1 + (u() * 2 - 1) * A['contraste']).to(x.device).view(B, 1, 1, 1)
    media = x.mean(dim=(1, 2, 3), keepdim=True)
    return (((x - media) * contraste + media) * brillo).clamp(0, 1)

def lote_normalizado(idx, gen=None):
    x = IMGS[idx].to(DISPOSITIVO).float() / 255
    if gen is not None:
        x = aumentar(x, gen)
    return (x - MEDIA_D) / DESV_D

def crear_cnn():
    m = timm.create_model('resnet50.a1_in1k', pretrained=True, num_classes=1, drop_rate=DROPOUT)
    torch.nn.init.zeros_(m.fc.weight); torch.nn.init.zeros_(m.fc.bias)
    for n, p in m.named_parameters():
        p.requires_grad = n.startswith(ENTRENABLES)
    return m.to(DISPOSITIVO)

def modo_entrenamiento(m):
    m.train()
    for n, c in m.named_children():
        if not n.startswith(ENTRENABLES):
            c.eval()                                   # capas congeladas con BatchNorm fijo

def entrenar_cnn(tr, semilla):
    torch.manual_seed(semilla); gen = torch.Generator().manual_seed(semilla)
    idx = torch.tensor(tr)
    sesiones = datos.iloc[tr].groupby('id_vaca')['bcs'].first()
    mu, sd = float(sesiones.mean()), float(sesiones.std())
    z = torch.tensor((Y[tr] - mu) / sd, dtype=torch.float32, device=DISPOSITIVO)
    w = torch.tensor(1 / datos.iloc[tr].groupby('id_vaca')['id_vaca'].transform('size').to_numpy(), dtype=torch.float32, device=DISPOSITIVO)
    m = crear_cnn()
    capas = [p for n, p in m.named_parameters() if p.requires_grad and not n.startswith('fc')]
    opt = torch.optim.AdamW([{'params': capas, 'lr': LR_CAPAS}, {'params': m.fc.parameters(), 'lr': LR_CABEZA}], weight_decay=WEIGHT_DECAY)
    pasos = len(tr) // LOTE
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCAS * pasos)
    escalador = torch.amp.GradScaler('cuda', enabled=USAR_AMP)
    for _ in range(EPOCAS):
        modo_entrenamiento(m)
        orden = torch.randperm(len(tr), generator=gen)
        for i in range(pasos):
            pos = orden[i * LOTE:(i + 1) * LOTE]
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USAR_AMP):
                salida = m(lote_normalizado(idx[pos], gen)).squeeze(1)
            wb, zb = w[pos.to(DISPOSITIVO)], z[pos.to(DISPOSITIVO)]
            perdida = (wb * (salida.float() - zb) ** 2).sum() / wb.sum()
            opt.zero_grad(set_to_none=True)
            escalador.scale(perdida).backward(); escalador.step(opt); escalador.update(); sched.step()
    return m, mu, sd

@torch.no_grad()
def predecir_cnn(m, te, mu, sd, lote=64):
    m.eval(); salida = []
    for i in range(0, len(te), lote):
        x = lote_normalizado(torch.tensor(te[i:i + lote]))
        salida.append(((m(x).squeeze(1) + m(torch.flip(x, dims=[3])).squeeze(1)) / 2).float().cpu())
    return mu + sd * torch.cat(salida).numpy()

def cnn_afinada(fold):
    pred = np.zeros(len(datos))
    for k in range(1, K_FOLDS + 1):
        tr, te = np.where(fold != k)[0], np.where(fold == k)[0]
        m, mu, sd = entrenar_cnn(tr, SEMILLA + k)
        pred[te] = predecir_cnn(m, te, mu, sd)
        del m
    return pred

if CORRER_E1:
    MODELOS = {'E1 CNN afinada': cnn_afinada, **MODELOS}

## 7. Validación cruzada con todas las semillas de folds

Cada foto se predice con un modelo que nunca vio a ese animal. Esto se repite para las 6 semillas de folds y las predicciones fuera de muestra se guardan en `PRED[semilla][modelo]`.

In [ ]:
PRED, inicio = {}, time.time()
for s in SEMILLAS_FOLDS:
    PRED[s] = {}
    for nombre, f in MODELOS.items():
        PRED[s][nombre] = f(FOLDS[s])
    print(f'semilla {s}: listo · {time.time() - inicio:.0f} s')

## 8. Métricas con los folds de la iteración anterior (semilla 42)

- Reportamos las métricas por foto y por vaca (promedio de las predicciones de cada sesión), en todo el rango de BCS.
- Pendiente de la recta predicho contra real: lo ideal es 1; un valor cercano a 0 significa que el modelo comprime todo hacia el promedio.
- Mejora contra E0 y E1: diferencia de MAE por vaca, con intervalo de confianza del 95 % por bootstrap por animal (se remuestrean animales completos y ambos modelos usan las mismas predicciones). Si el intervalo no incluye el 0, la diferencia es consistente.

In [ ]:
redondear = lambda x: np.clip(np.round(np.asarray(x, float) / PASO) * PASO, *ESCALA)

def metricas(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    r = redondear(p)
    return {'MAE': np.abs(p - y).mean(), 'RMSE': np.sqrt(((p - y) ** 2).mean()),
            'a_0.25_%': 100 * np.mean(np.abs(r - y) <= 0.25 + 1e-9),
            'spearman': spearmanr(y, p).correlation if np.std(p) > 0 else np.nan,
            'pendiente': np.polyfit(y, p, 1)[0]}

def por_sesion(pred):
    df = datos[['id_vaca', 'animal', 'bcs']].assign(**{m: v for m, v in pred.items()})
    return df.groupby('id_vaca').agg(animal=('animal', 'first'), bcs=('bcs', 'first'), **{m: (m, 'mean') for m in pred})

def ic_mejora(ses, modelo, referencia):
    g = ses.assign(er=(ses[referencia] - ses['bcs']).abs(), em=(ses[modelo] - ses['bcs']).abs(), n=1).groupby('animal')[['er', 'em', 'n']].sum()
    idx = np.random.default_rng(SEMILLA).integers(0, len(g), size=(BOOTSTRAP, len(g)))
    dif = (g['er'].to_numpy()[idx].sum(1) - g['em'].to_numpy()[idx].sum(1)) / g['n'].to_numpy()[idx].sum(1)
    return f'{np.percentile(dif, 2.5):+.3f} a {np.percentile(dif, 97.5):+.3f}'

pred = PRED[42]
ses = por_sesion(pred)
filas = []
for m in pred:
    mf, mv = metricas(Y, pred[m]), metricas(ses['bcs'], ses[m])
    fila = {'modelo': m, 'MAE_foto': mf['MAE'], 'RMSE_foto': mf['RMSE'], 'MAE_vaca': mv['MAE'], 'RMSE_vaca': mv['RMSE'],
            'a0.25_vaca_%': mv['a_0.25_%'], 'spearman_vaca': mv['spearman'], 'pendiente_vaca': mv['pendiente']}
    for ref in ['E0 constante', 'E1 CNN afinada']:
        if ref in pred and m != ref:
            fila[f'mejora_vs_{ref[:2]}'] = metricas(ses['bcs'], ses[ref])['MAE'] - mv['MAE']
            fila[f'IC95_vs_{ref[:2]}'] = ic_mejora(ses, m, ref)
    filas.append(fila)
tabla_42 = pd.DataFrame(filas).set_index('modelo').sort_values('MAE_vaca')
display(tabla_42.round(3))

## 9. Robustez entre semillas de folds

Como probamos varios modelos, el que sale mejor en la semilla 42 suele verse mejor de lo que realmente es. Las semillas 1 a 5 no se usaron para elegir nada, así que sirven como comprobación: si el orden de los modelos se mantiene en todas, la conclusión no depende de cómo se sortearon los folds.

In [ ]:
filas = []
for s in SEMILLAS_FOLDS:
    ses_s = por_sesion(PRED[s])
    for m in PRED[s]:
        filas.append({'semilla': s, 'modelo': m, 'MAE_vaca': metricas(ses_s['bcs'], ses_s[m])['MAE'],
                      'MAE_foto': metricas(Y, PRED[s][m])['MAE'], 'pendiente_vaca': metricas(ses_s['bcs'], ses_s[m])['pendiente']})
rob = pd.DataFrame(filas)
mae_semilla = rob.pivot(index='semilla', columns='modelo', values='MAE_vaca')
resumen = rob.groupby('modelo').agg(MAE_vaca=('MAE_vaca', 'mean'), sd_entre_semillas=('MAE_vaca', 'std'),
                                    MAE_foto=('MAE_foto', 'mean'), pendiente_vaca=('pendiente_vaca', 'mean'))
if 'E1 CNN afinada' in mae_semilla:
    resumen['semillas_que_gana_a_E1'] = (mae_semilla.lt(mae_semilla['E1 CNN afinada'], axis=0)).sum().astype(str) + f'/{len(SEMILLAS_FOLDS)}'
display(resumen.sort_values('MAE_vaca').round(3))
display(mae_semilla.round(3))

orden = resumen.sort_values('MAE_vaca').index
fig, ax = plt.subplots(figsize=(10, 4))
for i, m in enumerate(orden):
    ax.scatter(mae_semilla[m], [i] * len(mae_semilla), color='#2a78d6', alpha=0.6, s=30)
    ax.scatter(mae_semilla[m].mean(), i, color='#0b0b0b', marker='|', s=400)
ax.set(yticks=range(len(orden)), yticklabels=orden, xlabel='MAE por vaca (cada punto es una semilla de folds)')
ax.invert_yaxis(); ax.grid(axis='x', color='#e6e5e0'); ax.set_axisbelow(True)
for lado in ['top', 'right']:
    ax.spines[lado].set_visible(False)
ax.set_title('MAE por vaca en las 6 semillas de folds', loc='left', fontsize=10)
plt.tight_layout(); plt.show()

## 10. Criterios de éxito

En las fotos cuyo BCS real está entre 3.25 y 3.50 se pide MAE ≤ 0.25, RMSE ≤ 0.35 y al menos 80 % de predicciones a ±0.25 (con la predicción redondeada a 0.25). Los reportamos por semilla, porque un solo sorteo puede quedar justo arriba o justo abajo del umbral.

Ojo: predecir siempre 3.375 cumpliría los tres criterios sin aprender nada (MAE y RMSE de 0.125), porque en ese rango solo hay vacas de 3.25 y 3.50. Por eso la comparación principal es el MAE por vaca en todo el rango contra E0.

In [ ]:
en_rango = (Y >= INTERVALO[0]) & (Y <= INTERVALO[1])
filas = []
for s in SEMILLAS_FOLDS:
    for m in PRED[s]:
        r = metricas(Y[en_rango], PRED[s][m][en_rango])
        filas.append({'semilla': s, 'modelo': m, 'MAE': r['MAE'], 'RMSE': r['RMSE'], 'a_0.25_%': r['a_0.25_%'],
                      'cumple': r['MAE'] <= OBJ_MAE + 1e-9 and r['RMSE'] <= OBJ_RMSE + 1e-9 and r['a_0.25_%'] >= OBJ_TOLERANCIA - 1e-9})
crit = pd.DataFrame(filas)
display(crit.groupby('modelo').agg(MAE=('MAE', 'mean'), RMSE=('RMSE', 'mean'), a025=('a_0.25_%', 'mean'),
                                   semillas_que_cumplen=('cumple', 'sum')).sort_values('MAE').round(3))

## 11. Conclusiones (corrida local del 17/11/2026 en una Apple M2 Pro)

| Modelo | MAE por vaca (6 semillas) | MAE por foto | 3.25 a 3.50: MAE / % a ±0.25 | Gana a E1 |
|---|---|---|---|---|
| E3 DINOv2-L 448 + Ridge | 0.211 ± 0.008 | 0.243 | 0.237 / 80.6 % | 6 de 6 |
| E3 DINOv2-B 448 + Ridge | 0.230 ± 0.009 | 0.270 | 0.294 / 69.2 % | 6 de 6 |
| E3 DINOv2-L 224 + Ridge | 0.234 ± 0.006 | 0.254 | 0.239 / 81.3 % | 6 de 6 |
| E4 DINOv2-L 448 + MLP | 0.238 ± 0.010 | 0.259 | 0.251 / 78.4 % | 6 de 6 |
| E2 ResNet-50 congelada + Ridge | 0.266 ± 0.012 | 0.290 | 0.293 / 69.5 % | 5 de 6 |
| E1 CNN afinada | 0.273 ± 0.010 | 0.291 | 0.335 / 63.2 % | |
| E0 constante | 0.321 | 0.331 | 0.344 / 60.6 % | 0 de 6 |

1. DINOv2-L congelado a 448 px con Ridge es el mejor modelo en las 6 semillas de folds. Con los folds de la iteración anterior mejora a la CNN afinada por 0.071 de MAE por vaca (IC 95 %: 0.038 a 0.104) y a la constante por 0.106 (IC 95 %: 0.064 a 0.147).
2. Agregar capacidad entrenable no ayuda. La cabeza MLP (E4) queda por debajo de Ridge en todas las semillas, y afinar la ResNet-50 (E1) no supera a la misma red congelada con Ridge (E2). Con 80 animales funciona mejor un buen extractor congelado con un modelo lineal regularizado.
3. El tamaño y la resolución del extractor sí importan: DINOv2-B 224 < DINOv2-B 448 ≈ DINOv2-L 224 < DINOv2-L 448.
4. En los criterios de éxito, el mejor modelo los cumple en promedio (MAE 0.237, RMSE 0.290, 80.6 % a ±0.25), pero solo en 4 de las 6 semillas, así que está justo en el umbral.
5. Las predicciones siguen comprimidas hacia el promedio (pendiente de 0.43), sobre todo en las vacas flacas, de las que hay muy pocas fotos.